# Load the CLIP tokenizer, image processor and model

In [18]:
from transformers import CLIPTokenizer, CLIPImageProcessor, CLIPModel
from PIL import Image

model_id = "openai/clip-vit-base-patch32"
clip_tokenizer = CLIPTokenizer.from_pretrained(model_id)
clip_image_processor = CLIPImageProcessor.from_pretrained(model_id)
clip_model = CLIPModel.from_pretrained(model_id)


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

# Encode the images data

In [ ]:
import torch

imagePaths_data = ["data/food-101/images/chocolate_cake/13907.jpg", "data/food-101/images/chicken_wings/490.jpg"]
images_data = [Image.open(path).convert("RGB") for path in imagePaths_data] 

image_inputs_data = clip_image_processor(images=images_data, return_tensors="pt")
with torch.no_grad():
    image_features_data = clip_model.get_image_features(**image_inputs_data)  # [1, 512]
    if not isinstance(image_features_data, torch.Tensor):
        image_features_data = image_features_data.pooler_output
    image_features_data /= image_features_data.norm(dim=1, keepdim=True)

# Provide a search query (text)

In [20]:
query = ["chicken wings"]
text_inputs = clip_tokenizer(query, padding=True, return_tensors="pt")
with torch.no_grad():
    text_features = clip_model.get_text_features(**text_inputs)  # [3, 512]
    if not isinstance(text_features, torch.Tensor):
        text_features = text_features.pooler_output
    text_features /= text_features.norm(dim=1, keepdim=True)


# Compute similarities (text)

In [21]:
similarities = text_features @ image_features_data.T
temperature = clip_model.logit_scale.detach().exp()
rescaled_similarities = similarities * temperature
probabilities = torch.nn.functional.softmax(rescaled_similarities , dim=1)
probabilities


tensor([[3.2723e-06, 1.0000e+00]])

# Provide a search query (image)

In [ ]:
imagePaths_query = ["data/food-101/images/chocolate_cake/105067.jpg"]

images_query = [Image.open(path).convert("RGB") for path in imagePaths_query] 

image_inputs_query = clip_image_processor(images=images_query, return_tensors="pt")
with torch.no_grad():
    image_features_query = clip_model.get_image_features(**image_inputs_query)  # [1, 512]
    if not isinstance(image_features_query, torch.Tensor):
        image_features_query = image_features_query.pooler_output
    image_features_query /= image_features_query.norm(dim=1, keepdim=True)

# Compute similarities (image)

In [26]:
similarities = image_features_query @ image_features_data.T
temperature = clip_model.logit_scale.detach().exp()
rescaled_similarities = similarities * temperature
probabilities = torch.nn.functional.softmax(rescaled_similarities , dim=1)
probabilities

tensor([[1.0000e+00, 1.9374e-11]])